In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
data = pd.read_csv('diabetes.csv')

In [3]:
data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
data.shape

(768, 9)

In [5]:
data.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [6]:
from sklearn.ensemble import RandomForestClassifier

In [7]:
import warnings
warnings.filterwarnings('ignore')

In [8]:
data['Glucose'] = np.where(data['Glucose']==0, data['Glucose'].median(), data['Glucose'])

In [9]:
X = data.drop('Glucose', axis = 1)
y = data['Outcome']

In [10]:
data['Insulin'] = np.where(data['Insulin']==0, data['Insulin'].median(), data['Insulin'])

In [11]:
X.head()

,Pregnancies,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,72,35,0,33.6,0.627,50,1
1,1,66,29,0,26.6,0.351,31,0
2,8,64,0,0,23.3,0.672,32,1
3,1,66,23,94,28.1,0.167,21,0
4,0,40,35,168,43.1,2.288,33,1


In [12]:
data['SkinThickness'] = np.where(data['SkinThickness']==0, data['SkinThickness'].median(), data['SkinThickness'])

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=31)

In [15]:
rf_class = RandomForestClassifier(n_estimators = 10).fit(X_train, y_train)

In [16]:
pred  = rf_class.predict(X_test)

In [17]:
y.value_counts()

0    500
1    268
Name: Outcome, dtype: int64

In [18]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [19]:
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))
print(accuracy_score(y_test, pred))


[[98  0]
 [ 0 56]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        98
           1       1.00      1.00      1.00        56

    accuracy                           1.00       154
   macro avg       1.00      1.00      1.00       154
weighted avg       1.00      1.00      1.00       154

1.0


In [20]:
# let's start the hyper parameter tunning

### Manual Hyper Parameter Tunning

In [21]:
model=RandomForestClassifier(n_estimators=300,criterion='entropy',
                             max_features='sqrt',min_samples_leaf=10,random_state=100).fit(X_train,y_train)
predictions=model.predict(X_test)
print(confusion_matrix(y_test,predictions))
print(accuracy_score(y_test,predictions))
print(classification_report(y_test,predictions))

[[98  0]
 [ 0 56]]
1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        98
           1       1.00      1.00      1.00        56

    accuracy                           1.00       154
   macro avg       1.00      1.00      1.00       154
weighted avg       1.00      1.00      1.00       154



## Randomized Search CV
* Randomized search CV will pick the value randomly and do the predictions.
* Narrow down the results. 


In [22]:
from sklearn.model_selection import RandomizedSearchCV

In [23]:
# It's nothing but how many number of tree we want.
n_estimators = [int(i) for i in np.linspace(start = 100, stop = 2000, num = 10) ]

# Number of features we consider at very split
max_features = ['auto','sqrt','log2']

# Maximum number of tree in each tree
max_depth = [ int(i) for i in np.linspace(10,1000,10)]

# Minimum number of sample required at each node
min_sample_split = [1,2,3,4,6,8]

# Minimum number of sample required at each leaf
min_sample_leaf = [1,2,4,6,8]

random_grid = {'n_estimators': n_estimators,
              'max_features': max_features,
              'max_depth': max_depth,
              'min_samples_split': min_sample_split,
              'min_samples_leaf': min_sample_leaf,
              'criterion': ['entropy', 'gini']}

print(random_grid)



{'n_estimators': [100, 311, 522, 733, 944, 1155, 1366, 1577, 1788, 2000], 'max_features': ['auto', 'sqrt', 'log2'], 'max_depth': [10, 120, 230, 340, 450, 560, 670, 780, 890, 1000], 'min_samples_split': [1, 2, 3, 4, 6, 8], 'min_samples_leaf': [1, 2, 4, 6, 8], 'criterion': ['entropy', 'gini']}


In [24]:
rf = RandomForestClassifier()
rf_randomcv = RandomizedSearchCV(estimator = rf,param_distributions = random_grid, n_iter = 100, 
                          cv = 3, verbose = 2, random_state = 100, n_jobs = -1)
rf_randomcv.fit(X_train, y_train)

Fitting 3 folds for each of 100 candidates, totalling 300 fits


RandomizedSearchCV(cv=3, estimator=RandomForestClassifier(), n_iter=100,
                   n_jobs=-1,
                   param_distributions={'criterion': ['entropy', 'gini'],
                                        'max_depth': [10, 120, 230, 340, 450,
                                                      560, 670, 780, 890,
                                                      1000],
                                        'max_features': ['auto', 'sqrt',
                                                         'log2'],
                                        'min_samples_leaf': [1, 2, 4, 6, 8],
                                        'min_samples_split': [1, 2, 3, 4, 6, 8],
                                        'n_estimators': [100, 311, 522, 733,
                                                         944, 1155, 1366, 1577,
                                                         1788, 2000]},
                   random_state=100, verbose=2)

In [25]:
rf_randomcv.best_params_

{'n_estimators': 1577,
 'min_samples_split': 6,
 'min_samples_leaf': 1,
 'max_features': 'log2',
 'max_depth': 560,
 'criterion': 'gini'}

In [26]:
rf_randomcv.best_estimator_ # It's will give you the what values we need to take. 

RandomForestClassifier(max_depth=560, max_features='log2', min_samples_split=6,
                       n_estimators=1577)

In [27]:
best_params = rf_randomcv.best_estimator_

In [28]:
# Let's predict the data........
y_pred = best_paramscv.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))




NameError: name 'best_paramscv' is not defined

## Grid SearchCv

In [ ]:
rf_randomcv.best_params_ # This value we're going to use in GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    'criterion': [rf_randomcv.best_params_['criterion']],
    'max_depth': [rf_randomcv.best_params_['max_depth']],
    'max_features': [rf_randomcv.best_params_['max_features']],
    'min_samples_leaf': [rf_randomcv.best_params_['min_samples_leaf'], 
                         rf_randomcv.best_params_['min_samples_leaf']+2, 
                         rf_randomcv.best_params_['min_samples_leaf'] + 4],
    'min_samples_split': [rf_randomcv.best_params_['min_samples_split'] - 2,
                          rf_randomcv.best_params_['min_samples_split'] - 1,
                          rf_randomcv.best_params_['min_samples_split'], 
                          rf_randomcv.best_params_['min_samples_split'] +1,
                          rf_randomcv.best_params_['min_samples_split'] + 2],
    'n_estimators': [rf_randomcv.best_params_['n_estimators'] - 200, rf_randomcv.best_params_['n_estimators'] - 100, 
                     rf_randomcv.best_params_['n_estimators'], 
                     rf_randomcv.best_params_['n_estimators'] + 100, rf_randomcv.best_params_['n_estimators'] + 200]
}

print(param_grid)

In [ ]:
#n_jobs for just setting the cpu 
# verbose for just seeing the command or explanation below

In [ ]:
rf = RandomForestClassifier()
grid_search = GridSearchCV(estimator = rf, param_grid = param_grid, cv = 10, n_jobs = -1,
                          verbose = 2)
grid_search.fit(X_train, y_train)

In [ ]:
grid_search.best_estimator_

In [ ]:
grid_search.best_params_

In [ ]:
best_grid = grid_search.best_estimator_

In [ ]:
y_pred = best_grid.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


## Automated Hyperparameter Tunning
It has Three methods to perform hyperparameter Tunning
* Bayesian Optimization.
* Gradient Descent.
* Evolutionary Algorithms.

* First we see Bayesian Optimization.
* In that we see hyperopt.

In [ ]:
from hyperopt import hp, fmin, tpe, STATUS_OK, Trials

In [ ]:
space = {'criterion':  hp.choice('criterion', ['entropy','gini']),
         'max_depth':  hp.quniform('max_depth', 10, 1200, 10),
         'max_features': hp.choice('max_features', ['auto', 'sqrt', 'log2', None]),
         'min_samples_leaf': hp.uniform('min_samples_leaf', 0, 0.5),
         'min_samples_split': hp.uniform('min_samples_split', 0, 1), 
         'n_estimators': hp.choice('n_estimators', [10, 50, 100, 750, 120])}
# hp.choice is for taking anything what you have given, it's take like a choice.(when you have selection 
#----mechanism you can use this.)
# hp.quniform is select the integer values.
# hp.unifrom is select the floating numbers.


In [ ]:
space

In [ ]:
def objective(space):
    model = RandomForestClassifier(criterion = space['criterion'], max_depth =space['max_depth'],
                                  max_features = space['max_features'],
                                  min_samples_leaf = space['min_samples_leaf'],
                                  min_samples_split = space['min_samples_split'], 
                                  n_estimators = space['n_estimators']
                                  )
    accuracy = cross_val_score(model, X_train, y_train, cv = 5).mean()
    return{'loss': -accuracy, 'status': STATUS_OK}

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
trials = Trials()
best = fmin(fn = objective,
           space = space, 
           algo = tpe.suggest,
           max_evals = 80,
           trials = trials)
best

In [ ]:
crit = {0: 'entropy', 1: 'gini'}
feat = {0: 'auto', 1: 'sqrt', 2: 'log2', 3: None}
est = {0: 10, 1: 50, 2: 100, 3: 750, 4: 120}


print(crit[best['criterion']])
print(feat[best['max_features']])
print(est[best['n_estimators']])
# we are just mapping

In [ ]:
trainedforest = RandomForestClassifier(criterion = crit[best['criterion']], max_depth = best['max_depth'], 
                                       max_features = feat[best['max_features']], 
                                       min_samples_leaf = best['min_samples_leaf'], 
                                       min_samples_split = best['min_samples_split'], 
                                       n_estimators = est[best['n_estimators']]).fit(X_train,y_train)
predictionforest = trainedforest.predict(X_test)
print(confusion_matrix(y_test,predictionforest))
print(accuracy_score(y_test,predictionforest))
print(classification_report(y_test,predictionforest))
acc5 = accuracy_score(y_test,predictionforest)

### Genetic Algorithm
* i) It's nothing but we can give N number of models, based on the models which models are perform better that was given by the list. 
* ii) Based on that we will create many offsprings(childs) models, we will create multiple models with same hyperparameter.
* adv: It's works well.
*      It will create a offspring for those model. 

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)]
# Number of features to consider at every split
max_features = ['auto', 'sqrt','log2']
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 1000,10)]
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10,14]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4,6,8]
# Create the random grid
param = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
              'criterion':['entropy','gini']}
print(param)

In [ ]:
from tpot import TPOTClassifier

In [ ]:
tpot_classifier = TPOTClassifier(generations =5, population_size = 24, offspring_size = 12, 
                                verbosity = 2, early_stop = 12,
                                config_dict = {'sklearn.ensemble.RandomForestClassifier': param}, 
                                cv = 4, scoring = 'accuracy')
tpot_classifier.fit(X_train, y_train)
# Generation is how may times you need like epochs.
# population is size is how many models  needed.
# offspring_size in the population how many model you need.



In [ ]:
tpot_classifier.get_params

In [ ]:
accuracy = tpot_classifier.score(X_test, y_test)
accuracy

### optuna


In [ ]:
import sklearn.svm
def objective(trial):

    classifier = trial.suggest_categorical('classifier', ['RandomForest', 'SVC'])
    
    if classifier == 'RandomForest':
        n_estimators = trial.suggest_int('n_estimators', 200, 2000,10)
        max_depth = int(trial.suggest_float('max_depth', 10, 100, log=True))

        clf = sklearn.ensemble.RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth)
    else:
        c = trial.suggest_float('svc_c', 1e-10, 1e10, log=True)
        
        clf = sklearn.svm.SVC(C=c, gamma='auto')

    return sklearn.model_selection.cross_val_score(
        clf,X_train,y_train, n_jobs=-1, cv=3).mean()


In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

trial = study.best_trial

print('Accuracy: {}'.format(trial.value))
print("Best hyperparameters: {}".format(trial.params))

In [ ]:
import optuna

In [ ]:
study.best_params

In [ ]:
rf=RandomForestClassifier(n_estimators=330,max_depth=30)
rf.fit(X_train,y_train)

In [ ]:
y_pred=rf.predict(X_test)
print(confusion_matrix(y_test,y_pred))
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

In [ ]:
from sklearn.ensemble  import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier()

# Hyper Parameter Optimization for Decision Trees

In [ ]:
params = { 
   'splitter' : ['best', 'random'], 
   'max_depth': [3,4,5,2,6,26,2,1,12,14],
    'min_samples_leaf': [1,2,3,4,5],
    'min_weight_fraction_leaf': [0.1, 0.2, 0.3, 0.4], 
    'max_features': ['auto', 'log2', 'sqrt', None],
    'max_leaf_nodes': [None, 10, 20, 30, 40, 50, 60]
    
    }

In [ ]:
params

# Hyper Parameter Optimization for XGB

In [ ]:
import numpy as np

In [ ]:
# Randomized Search Cv

# nr of trees in ra
n_estimators = [int(x) for x in np.linspace(100, 1200, 12)]

# various learning rate parametersa
learning_rate = ['0.05', '0.1', '0.3', '0.4', '0.5', '0.6']

# Maximum nr of levles in trees.
max_depth = [int(x) for x in np.linspace(5, 30, num =6)]

#subsample parameters values 
subsample = [0.7, 0.6, 0.8]

# Minimum Child weight parameter
min_child_weight = [3,4,5,6,7]


In [ ]:
random_grid = { 'n_estimators': n_estimators, 
              'learning_rate': learning_rate,
              'max_depth': max_depth,
              'subsample': subsample,
              'min_child_weight': min_child_weight}
print(random_grid)

# Hyperparamter optimization for KNN

In [ ]:
accuracy_rate = []

for i in range(1,40):
    kn = KNeighborRegressor(n_neighbor = i)
    score = cross_val_score(knn, X, y, cv = 10, scoring = 'neg_mean_squarred_error')
    accuracy_rate.append(score.mean())
    

In [ ]:
plt.figure(figsize = (10,y))
plt.plot(range(1,40), accuracy_rate)
# The sudden decrease in the line we can take as a neighbor value. 